## Prova delle funzioni baselines

### Setup

In [ ]:
#metriche d'errore
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
#per la regression
from sklearn.linear_model import LinearRegression
#per il gradient boosting
from xgboost import XGBRegressor

In [ ]:
from data_pipeline import *

#split and clean
df = load_data('data/data-ready-def.xlsx')
train_df, valid_df, test_df = split_data(df)  #i tre set splittati

train_df_clean = clean_data(train_df)
valid_df_clean = clean_data(valid_df)
test_df_clean = clean_data(test_df)

print(train_df_clean.shape)
print(valid_df_clean.shape)
print(test_df_clean.shape)

print((train_df_clean['Amount of irrigation'] < 0).sum())
print((valid_df_clean['Amount of irrigation'] < 0).sum())
print((test_df_clean['Amount of irrigation'] < 0).sum())

In [ ]:
#normalize
train_df_norm, valid_df_norm, test_df_norm, train_df_min, train_df_max = normalize(train_df_clean, valid_df_clean, test_df_clean)

columns = train_df_norm.columns.drop(['Date', 'year'])

print(train_df_norm[columns].min())
print(train_df_norm[columns].max())
print('\n\n')
print(valid_df_norm[columns].min())
print(valid_df_norm[columns].max())
print('\n\n')
print(test_df_norm[columns].min())
print(test_df_norm[columns].max())

In [ ]:
#create_sequence
train_X, train_y = create_sequences(train_df_norm, 20, 1)
print(train_X.shape)
print(train_y.shape)

valid_X, valid_y = create_sequences(valid_df_norm, 20, 1)
print(valid_X.shape)
print(valid_y.shape)

test_X, test_y = create_sequences(test_df_norm, 20, 1)
print(test_X.shape)
print(test_y.shape)

### Climatology : "cosa è normale per questo periodo della stagione"

Calcoli, per ciascun giorno della stagione (giorno 1, giorno 2, ..., giorno 153), la media storica
di irrigazione in quel giorno specifico, usando solo gli anni di training.
Poi, per qualsiasi giorno di validation/test, predici semplicemente quella media storica — ignorando completamente
il meteo specifico di quel giorno/anno.
Cattura il pattern stagionale (più irrigazione in piena estate, meno agli estremi della stagione)
ma ignora tutto il resto. È la previsione più 'ignorante' che abbia comunque senso.

In [ ]:
#numero, per ogni stagione (anno), i giorni da 0 a 152, aggiungo 1 così da avere 1-153  
df['day of season'] = df.groupby('year').cumcount() + 1
print(df['day of season'])

In [ ]:
#numero i giorni dei tre set (NOTA: lavoro con i set non normalizzati, poiché Climatology è una semplice media di valori,
#vale lo stesso anche per Persistence e Seasonal Naïve)
train_df_clean['day of season'] = train_df_clean.groupby('year').cumcount() + 1
valid_df_clean['day of season'] = valid_df_clean.groupby('year').cumcount() + 1
test_df_clean['day of season'] = test_df_clean.groupby('year').cumcount() + 1

print(train_df_clean['day of season'])
print(valid_df_clean['day of season'])
print(test_df_clean['day of season'])

In [ ]:
#calcolo la media di irrigazione per tutti i giorni di ogni anno del train
climatology = train_df_clean.groupby('day of season')['Amount of irrigation'].mean()
print(climatology)

In [ ]:
#'predico' i valori di climatology di tutti i giorni del valid e del test set
#per esempio, se nel giorno 1 del train ho 0.11, sostituisco questo valore nello stesso giorno del valid e del test set, per ogni anno
valid_pred_climatology = valid_df_clean['day of season'].map(climatology) #riportano le previsioni
test_pred_climatology = test_df_clean['day of season'].map(climatology)

print(valid_pred_climatology)
print(test_pred_climatology)

In [ ]:
#calcolo le metriche per il calcolo degli errori

#prima valore reale, poi previsione
valid_mae = mean_absolute_error(valid_df_clean['Amount of irrigation'], valid_pred_climatology)
valid_rmse = np.sqrt(mean_squared_error(valid_df_clean['Amount of irrigation'], valid_pred_climatology))
valid_r2 = r2_score(valid_df_clean['Amount of irrigation'], valid_pred_climatology)

print(valid_mae, valid_rmse, valid_r2)

#salvo questi valori per il calcolo futuro dello skill score
climatology_valid_mae = valid_mae
climatology_valid_rmse = valid_rmse
climatology_valid_r2 = valid_r2

test_mae = mean_absolute_error(test_df_clean['Amount of irrigation'], test_pred_climatology)
test_rmse = np.sqrt(mean_squared_error(test_df_clean['Amount of irrigation'], test_pred_climatology))
test_r2 = r2_score(test_df_clean['Amount of irrigation'], test_pred_climatology)

climatology_test_mae = test_mae
climatology_test_rmse = test_rmse
climatology_test_r2 = test_r2

print(test_mae, test_rmse, test_r2)


### Persistence: "oggi sarà come H giorni fa"

Letteralmente copi il valore di H giorni prima come previsione per oggi.
Funziona bene quando le condizioni non cambiano bruscamente di giorno in giorno (il meteo è autocorrelato),
specialmente con H piccolo. È sorprendentemente difficile da battere per orizzonti brevi
— motivo per cui è un classico baseline nel forecasting meteorologico.

In [ ]:
group_col = 'year'
target_col = 'Amount of irrigation'
H = 1

valid_pred_persistence = valid_df_clean.groupby(group_col)[target_col].shift(H)
test_pred_persistence = test_df_clean.groupby(group_col)[target_col].shift(H)

print(valid_pred_persistence)
print(test_pred_persistence)

In [ ]:
#filtro i valori diversi da NaN e li salvo
valid_mask = valid_pred_persistence.notna()
test_mask = test_pred_persistence.notna()

#metriche
valid_mae = mean_absolute_error(valid_df_clean['Amount of irrigation'][valid_mask], valid_pred_persistence[valid_mask])
valid_rmse = np.sqrt(mean_squared_error(valid_df_clean['Amount of irrigation'][valid_mask], valid_pred_persistence[valid_mask]))
valid_r2 = r2_score(valid_df_clean['Amount of irrigation'][valid_mask], valid_pred_persistence[valid_mask])

print(valid_mae, valid_rmse, valid_r2)

#salvo questi valori per il calcolo futuro dello skill score
persistence_valid_mae = valid_mae
persistence_valid_rmse = valid_rmse
persistence_valid_r2 = valid_r2

test_mae = mean_absolute_error(test_df_clean['Amount of irrigation'][test_mask], test_pred_persistence[test_mask])
test_rmse = np.sqrt(mean_squared_error(test_df_clean['Amount of irrigation'][test_mask], test_pred_persistence[test_mask]))
test_r2 = r2_score(test_df_clean['Amount of irrigation'][test_mask], test_pred_persistence[test_mask])

persistence_test_mae = test_mae
persistence_test_rmse = test_rmse
persistence_test_r2 = test_r2

print(test_mae, test_rmse, test_r2)

In [ ]:
#calcolo skill score Persistence/Climatology -> se valori positivi, Persistence ha meno MAE

skill_persistence_valid = 1 - (persistence_valid_mae/climatology_valid_mae)
skill_persistence_test = 1 - (persistence_test_mae/climatology_test_mae)

print(skill_persistence_valid)
print(skill_persistence_test)

### Seasonal Naïve: "quest'anno sarà come l'anno scorso, nello stesso punto della stagione"

Predici usando il valore di esattamente una stagione prima (153 giorni prima) nello stesso giorno-di-stagione.
Cattura la ripetizione anno-su-anno, ma non si adatta se quell'anno specifico è stato
anomalo (più piovoso o più secco della norma).

In [ ]:
target_col = 'Amount of irrigation'
season_length = 153

#scambio i valori di irrigazione odierni con quelli di un anno (153 giorni) fa
#i primi anni di ogni set non hanno un precedente, stessa gestione valori NaN
valid_pred_seasonal = valid_df_clean[target_col].shift(season_length)
test_pred_seasonal = test_df_clean[target_col].shift(season_length)

print(valid_pred_seasonal)
print(test_pred_seasonal)

#valori NaN in ogni set (anni precedenti a quelli iniziali)
print(valid_pred_seasonal.isna().sum())
print(test_pred_seasonal.isna().sum())

In [ ]:
#calcolo metriche
valid_mask = valid_pred_seasonal.notna()
test_mask = test_pred_seasonal.notna()

valid_mae = mean_absolute_error(valid_df_clean[target_col][valid_mask], valid_pred_seasonal[valid_mask])
valid_rmse = np.sqrt(mean_squared_error(valid_df_clean[target_col][valid_mask], valid_pred_seasonal[valid_mask]))
valid_r2 = r2_score(valid_df_clean[target_col][valid_mask], valid_pred_seasonal[valid_mask])

print(valid_mae, valid_rmse, valid_r2)

#salvo questi valori per il calcolo futuro dello skill score
seasonal_valid_mae = valid_mae 
seasonal_valid_rmse = valid_rmse
seasonal_valid_r2 = valid_r2

test_mae = mean_absolute_error(test_df_clean[target_col][test_mask], test_pred_seasonal[test_mask])
test_rmse = np.sqrt(mean_squared_error(test_df_clean[target_col][test_mask], test_pred_seasonal[test_mask]))
test_r2 = r2_score(test_df_clean[target_col][test_mask], test_pred_seasonal[test_mask])

print(test_mae, test_rmse, test_r2)

#salvo questi valori per il calcolo futuro dello skill score
seasonal_test_mae = test_mae
seasonal_test_rmse = test_rmse
seasonal_test_r2 = test_r2

In [ ]:
#skill score Seasonal/Climatology

skill_seasonal_valid = 1 - (seasonal_valid_mae/climatology_valid_mae)
skill_seasonal_test = 1 - (seasonal_test_mae/climatology_test_mae)

print(skill_seasonal_valid)
print(skill_seasonal_test)

### Linear Regression: modello statistico

Impara una combinazione lineare pesata delle feature meteo per prevedere il target.
Se questo modello semplice arriva già vicino alle prestazioni del deep model, vuol dire che la
relazione meteo→irrigazione è abbastanza lineare da non richiedere reti neurali.

In [ ]:
#per questa baseline utilizzo i 3 set normalizzati e già come sequenze temporali (create sequences)
#al momento i tre set sono in 3D, ma la Linear Regression lavora in 2D, quindi appiattiamoli in questo modo:

train_X_flat = train_X.reshape(train_X.shape[0], -1)  # (2261, 160)
valid_X_flat = valid_X.reshape(valid_X.shape[0], -1)
test_X_flat = test_X.reshape(test_X.shape[0], -1)

#addestro il modello prima sul train
lr_model = LinearRegression()
lr_model.fit(train_X_flat, train_y)

#predizioni su valid e test (valori ancora normalizzati, mi servono isodimensionali a quelli delle altre baselines)
valid_pred_norm = lr_model.predict(valid_X_flat)
test_pred_norm = lr_model.predict(test_X_flat)

#denormalizzo le predizioni
target_min = train_df_min['Amount of irrigation']
target_max = train_df_max['Amount of irrigation']

#li rendo in mm
valid_pred_mm = valid_pred_norm * (target_max - target_min) + target_min
valid_true_mm = valid_y * (target_max - target_min) + target_min  #valori reali in mm

test_pred_mm = test_pred_norm * (target_max - target_min) + target_min
test_true_mm = test_y * (target_max - target_min) + target_min  #valori reali in mm

valid_pred_mm = np.clip(valid_pred_mm, a_min=0, a_max=None)
test_pred_mm = np.clip(test_pred_mm, a_min=0, a_max=None)

#ora posso calcolare le metriche
lr_valid_mae = mean_absolute_error(valid_true_mm, valid_pred_mm)
lr_valid_rmse = np.sqrt(mean_squared_error(valid_true_mm, valid_pred_mm))
lr_valid_r2 = r2_score(valid_true_mm, valid_pred_mm)

lr_test_mae = mean_absolute_error(test_true_mm, test_pred_mm)
lr_test_rmse = np.sqrt(mean_squared_error(test_true_mm, test_pred_mm))
lr_test_r2 = r2_score(test_true_mm, test_pred_mm)

print(lr_valid_mae, lr_valid_rmse, lr_valid_r2)
print(lr_test_mae, lr_test_rmse, lr_test_r2)

In [ ]:
#skill score lr/Climatology
skill_lr_valid = 1 - (lr_valid_mae/climatology_valid_mae)
skill_lr_test = 1 - (lr_test_mae/climatology_test_mae)

print(skill_lr_valid)
print(skill_lr_test)

### Gradient Boosting (XGBoost)

Cattura relazioni non lineari e interazioni tra feature senza essere una rete neurale.
Di solito è il più difficile da battere tra i 5 — se il tuo modello deep non supera XGBoost
con un margine chiaro, è difficile giustificare la complessità aggiuntiva del deep learning per questo problema.

In [ ]:
gb_model = XGBRegressor(random_state=42)

#addestro il modello sul train
gb_model.fit(train_X_flat, train_y)

#predico su valid e set
valid_pred_norm_gb = gb_model.predict(valid_X_flat)
test_pred_norm_gb = gb_model.predict(test_X_flat)

#denormalizzo i valori rendendoli in mm
valid_pred_mm_gb = valid_pred_norm_gb * (target_max - target_min) + target_min
test_pred_mm_gb = test_pred_norm_gb * (target_max - target_min) + target_min

#rendo nulli i valori di irrigazione negativa
valid_pred_mm_gb = np.clip(valid_pred_mm_gb, a_min=0, a_max=None)
test_pred_mm_gb = np.clip(test_pred_mm_gb, a_min=0, a_max=None)

#i valori reali (_true_mm) sono comuni a quelli della lr, non serve ricalcolarli

#calcolo le metriche
gb_valid_mae = mean_absolute_error(valid_true_mm, valid_pred_mm_gb)
gb_valid_rmse = np.sqrt(mean_squared_error(valid_true_mm, valid_pred_mm_gb))
gb_valid_r2 = r2_score(valid_true_mm, valid_pred_mm_gb)

gb_test_mae = mean_absolute_error(test_true_mm, test_pred_mm_gb)
gb_test_rmse = np.sqrt(mean_squared_error(test_true_mm, test_pred_mm_gb))
gb_test_r2 = r2_score(test_true_mm, test_pred_mm_gb)

print(gb_valid_mae, gb_valid_rmse, gb_valid_r2)
print(gb_test_mae, gb_test_rmse, gb_test_r2)

In [ ]:
#skill score gb/Climatology
skill_gb_valid = 1 - (gb_valid_mae/climatology_valid_mae)
skill_gb_test = 1 - (gb_test_mae/climatology_test_mae)

print(skill_gb_valid)
print(skill_gb_test)